In [1]:
import numpy as np
import cv2
import os
from PIL import Image, ImageDraw
from skimage.feature import graycomatrix, graycoprops, hog
from skimage.color import rgb2gray
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import RandomOverSampler
from collections import Counter
from sklearn.utils import shuffle
from sklearn import svm
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, StackingClassifier, VotingClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import xgboost as xgb
import pandas as pd
from sklearn.model_selection import train_test_split

# Evaluation Metrics
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics.pairwise import cosine_similarity

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# _________________________________________Load Feature Extraction Vectores ________________________________
# ____________________________________ hog or glcm or orb , vgg , yolo , resnet ____________________________


In [2]:
# load Feature Extraction Vectors (hog, glcm, orb, vgg, yolo, resnet, squeezenet, ML, DL, ALL)
import numpy as np

# Set feature_type to one of: 'all', 'ml', 'dl', 'hog', 'glcm', 'orb', 'vgg', 'yolo', 'squeezenet', 'resnet'
feature_type = 'yolo'  # Change to 'all', 'ml', 'dl', 'hog', 'glcm', 'orb', 'vgg', 'yolo', 'squeezenet', 'resnet' as needed

features_array = None

if feature_type == 'all':
    df = pd.read_csv('Bin_Features/combined_all_features_array.csv')
    print("Loaded ALL features (ML + DL).")
elif feature_type == 'ml':
    df = pd.read_csv('Bin_Features/combined_ML_features_array.csv')
    print("Loaded ML features.")
elif feature_type == 'dl':
    df = pd.read_csv('Bin_Features/combined_DL_features_array.csv')
    print("Loaded DL features.")
elif feature_type == 'hog':
    df = pd.read_csv('Bin_Features/hog_features_array.csv')
    print("Loaded HOG features.")
elif feature_type == 'glcm':
    df = pd.read_csv('Bin_Features/glcm_features_array.csv')
    print("Loaded GLCM features.")
elif feature_type == 'orb':
    df = pd.read_csv('Bin_Features/orb_features_array.csv')
    print("Loaded ORB features.")
elif feature_type == 'vgg':
    df = pd.read_csv('Bin_Features/vgg_features_array.csv')
    print("Loaded VGG features.")
elif feature_type == 'resnet':
    df = pd.read_csv('Bin_Features/resnet_features_array.csv')
    print("Loaded ResNet features.")
elif feature_type == 'squeezenet':
    df = pd.read_csv('Bin_Features/squeezenet_features_array.csv')
    print("Loaded SqueezeNet features.")
elif feature_type == 'yolo':
    df = pd.read_csv('Bin_Features/yolo_features_array.csv')
    print("Loaded YOLO features.")
else:
    raise ValueError("Invalid feature_type. Choose from 'all', 'ml', 'dl', 'hog', 'glcm', 'orb', 'vgg', 'yolo', 'squeezenet', 'resnet'.")
# Separate features and labels
features_array = df.drop('label', axis=1).values
labels_array = df['label'].values

print('Features shape:', features_array.shape)
print('Labels shape:', labels_array.shape)

print(f"------++++++++ Load Feature Extraction Vectors is Done :- {feature_type} +++++++++-------------")

Loaded YOLO features.
Features shape: (6390, 128)
Labels shape: (6390,)
------++++++++ Load Feature Extraction Vectors is Done :- yolo +++++++++-------------


In [3]:
from data_processing import classwise_test_replacement


In [4]:

new_data = classwise_test_replacement(
    df.values,
    train_size=6000,
    ratio=0,
    random_state=42
)

features_array = new_data[:, :-1]
labels_array = new_data[:, -1]


In [5]:
import numpy as np

def split_train_test(data, train_size=6000):
    """
    Split dataset into train and test sets.

    Parameters:
    -----------
    data : np.ndarray
        Full dataset
    train_size : int
        Number of rows for training

    Returns:
    --------
    X_train, y_train, X_test, y_test
    """

    train = data[:train_size]
    test = data[train_size:]

    X_train = train[:, :-1]
    y_train = train[:, -1]

    X_test = test[:, :-1]
    y_test = test[:, -1]

    return X_train, y_train, X_test, y_test

In [6]:
from sklearn.model_selection import train_test_split

X_train, y_train, X_test, y_test = split_train_test(new_data,train_size=6000)

# Print the shapes of the train and test sets

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (6000, 128)
X_test shape: (390, 128)
y_train shape: (6000,)
y_test shape: (390,)


In [7]:
from imblearn.over_sampling import RandomOverSampler

# Oversample the features_array and labels to balance the classes
oversample = RandomOverSampler(sampling_strategy='not majority', random_state=42)
X_train2, y_train2 = oversample.fit_resample(X_train, y_train)

print("Original class distribution:", Counter(y_train))
print("Resampled class distribution:", Counter(y_train2))
print("Resampled features_array shape:", X_train2.shape)

Original class distribution: Counter({'osteoporosis': 3590, 'normal': 2410})
Resampled class distribution: Counter({'osteoporosis': 3590, 'normal': 3590})
Resampled features_array shape: (7180, 128)


In [8]:

# Feature selection ---- selected_features_glcm
# Initialize the classifier
# Define the number of features to select
#from hormany_selection import harmony_search_feature_selection
from harmony_feature_selection import harmony_search_feature_selection

selected_features_comb, final_fitness = harmony_search_feature_selection(
    X_train2,
    y_train2,
    num_agents=20,
    max_iter=100,
    HMCR=0.9,
    PAR=0.3,

    min_features=10,
    early_stopping=10
)


print("Selected features_glcm (binary vector):", selected_features_comb)




Evaluating initial population (20 agents) using 2 cores...
Starting Optimization Loop...
Iteration 10/100 | Best Fitness: 0.9087 | Features: 64
Iteration 20/100 | Best Fitness: 0.9087 | Features: 64

Stopping early - No improvement for 10 iterations

Feature selection completed.
Best Fitness: 0.9087
Number of selected features: 64
Selected feature indices: [  0   1   5   6   8  11  14  20  23  24  26  27  29  30  32  36  40  41
  42  43  45  47  48  50  52  56  58  61  62  63  64  69  71  72  73  74
  77  80  82  84  85  91  92  94  95  97  99 100 101 102 103 106 107 108
 111 114 115 116 119 121 122 124 125 126]
Selected features_glcm (binary vector): [1 1 0 0 0 1 1 0 1 0 0 1 0 0 1 0 0 0 0 0 1 0 0 1 1 0 1 1 0 1 1 0 1 0 0 0 1
 0 0 0 1 1 1 1 0 1 0 1 1 0 1 0 1 0 0 0 1 0 1 0 0 1 1 1 1 0 0 0 0 1 0 1 1 1
 1 0 0 1 0 0 1 0 1 0 1 1 0 0 0 0 0 1 1 0 1 1 0 1 0 1 1 1 1 1 0 0 1 1 1 0 0
 1 0 0 1 1 1 0 0 1 0 1 1 0 1 1 1 0]


In [9]:
# Apply selected features on the entire dataset
# يتم تطيبق الصفات المختارة على جميع البيانات ( بيانات التدريب والاختبار ) البيانات التي الحصول عليها من استخلاص الصفات
from harmony_feature_selection import apply_selected_features

X_selected_comb = apply_selected_features(features_array, selected_features_comb)

print('feature shape befor FS :', features_array.shape)
print('feature shape After FS :', X_selected_comb.shape)

# Combine features and labels into a DataFrame
df = pd.DataFrame(X_selected_comb)
df['label'] = labels_array

# Save to CSV

if feature_type == 'all':
    df.to_csv('hormany_FS/all_features_array.csv', index=False)
    print("Save ML-DL .")
elif feature_type == 'ml':
    df.to_csv('hormany_FS/ml_features_array.csv', index=False)
    print("Save ML feature .")
elif feature_type == 'dl':
    df.to_csv('hormany_FS/dl_features_array.csv', index=False)
    print("Save DL feature .")
elif feature_type == 'hog':
    df.to_csv('hormany_FS/hog_features_array.csv', index=False)
    print("Save hog_hormany.")
elif feature_type == 'glcm':
    df.to_csv('hormany_FS/glcm_features_array.csv', index=False)
    print("Save glcm_hormany.")
elif feature_type == 'orb':
    df.to_csv('hormany_FS/orb_features_array.csv', index=False)
    print("Save orb_hormany.")
elif feature_type == 'vgg':
    df.to_csv('hormany_FS/vgg_features_array.csv', index=False)
    print("Save vgg_hormany.")
elif feature_type == 'resnet':
    df.to_csv('hormany_FS/resnet_features_array.csv', index=False)
    print("Save resnet_hormany.")
elif feature_type == 'squeezenet':
    df.to_csv('hormany_FS/squeezenet_features_array.csv', index=False)
    print("Save squeezenet_hormany.")
elif feature_type == 'yolo':
    df.to_csv('hormany_FS/yolo_features_array.csv', index=False)
    print("Save yolo_hormany.")
else:
    raise ValueError("Invalid feature_type. Choose from 'all', 'hog', 'glcm', 'orb', 'vgg', 'squeezenet', 'resnet', 'yolo'.")




feature shape befor FS : (6390, 128)
feature shape After FS : (6390, 64)
Save yolo_hormany.


In [10]:
from sklearn.model_selection import train_test_split

X_train, y_train, X_test, y_test = split_train_test(df.values, train_size=6000)

# Print the shapes of the train and test sets

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (6000, 64)
X_test shape: (390, 64)
y_train shape: (6000,)
y_test shape: (390,)


# __________________________________ Test Stage _____________________________________________
# _________________________Load Feature Extraction Vectores After Feature selection________________________________
# ____________________________________ hog or glcm or orb , vgg , yolo , resnet ____________________________


%%%%%%%%%%%%%%%%%%%%%%%%% Test without Cross Validation %%%%%%%%%%%%%%%%%%%%%%%%%%%%%

In [17]:
# === إصلاح شامل لمشكلة الأنواع ===
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

# 1. نقوم بتدريب المشفر على y_train2 (بعد التضخيم) لأنه يحتوي على جميع الأصناف
le.fit(y_train)

# 2. نقوم بتحويل جميع متغيرات التسميات المستخدمة في الكود
# (هذا هو السطر الذي كان مفقوداً ويسبب الخطأ)
y_train = le.transform(y_train)

# تحويل المتغيرات الأخرى
#y_train2 = le.transform(y_train2)
y_test = le.transform(y_test)

print("تم تحويل جميع المتغيرات (y_train, y_train2, y_test) إلى أرقام بنجاح.")
print(f"الأصناف: {le.classes_}")

تم تحويل جميع المتغيرات (y_train, y_train2, y_test) إلى أرقام بنجاح.
الأصناف: [0 1]


In [ ]:
from sklearn.ensemble import (
    RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier,
    ExtraTreesClassifier, BaggingClassifier, VotingClassifier,
    HistGradientBoostingClassifier # إضافة الوحدات الجديدة
)
from sklearn.linear_model import (
    LogisticRegression, RidgeClassifier, SGDClassifier,
    PassiveAggressiveClassifier, Perceptron
)
from sklearn.svm import SVC, NuSVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier, RadiusNeighborsClassifier
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB, ComplementNB
from sklearn.tree import DecisionTreeClassifier, ExtraTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.semi_supervised import LabelPropagation, LabelSpreading
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier # إضافة الوحدة الجديدة
from collections import Counter
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# التحقق من تثبيت CatBoost
try:
    from catboost import CatBoostClassifier
    CATBOOST_AVAILABLE = True
except ImportError:
    CATBOOST_AVAILABLE = False
    print("Warning: CatBoost not installed. Run '!pip install catboost'")

# Find the most frequent label for outlier_label
most_frequent_label = Counter(y_train).most_common(1)[0][0]



classifiers = {
    # --- المصنفات القوية (مفعلة) ---
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
    'ExtraTrees': ExtraTreesClassifier(n_estimators=100, random_state=42),
    # 'AdaBoost': AdaBoostClassifier(n_estimators=50, random_state=42), # تم إيقافه (ضعيف)
    # 'GradientBoosting': GradientBoostingClassifier(n_estimators=10, random_state=42), # تم إيقافه (تم استبداله)
    # 'Bagging': BaggingClassifier(n_estimators=10, random_state=42), # تم إيقافه
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    #'SVM': SVC(random_state=42, probability=True),
    # 'KNN': KNeighborsClassifier(n_neighbors=5), # تم إيقافه (بطيء مع 5000 ميزة)
    # 'GaussianNB': GaussianNB(), # تم إيقافه (ضعيف مع البيانات المرتبطة)
    # 'DecisionTree': DecisionTreeClassifier(random_state=42), # تم إيقافه (ضعيف بمفرده)
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss'),
    'MLPClassifier': MLPClassifier(random_state=42, max_iter=500),
    'LDA': LinearDiscriminantAnalysis(),
    'LinearSVC': LinearSVC(random_state=42, max_iter=200),
    # 'QDA': QuadraticDiscriminantAnalysis(), # تم إيقافه (يفشل مع الأبعاد العالية)

    # --- المصنفات الجديدة المضافة (مفعلة) ---
    'LightGBM': LGBMClassifier(random_state=42, verbose=-1, n_jobs=-1),
    'HistGradientBoosting': HistGradientBoostingClassifier(max_iter=100, random_state=42),
    'RidgeClassifier': RidgeClassifier(random_state=42),
}

# إضافة CatBoost إذا كان متاحاً
#if CATBOOST_AVAILABLE:
 #   classifiers['CatBoost'] = CatBoostClassifier(iterations=100, learning_rate=0.1, depth=6, verbose=False, random_state=42)


results = {}

metrics_table = []
for name, clf in classifiers.items():
    print(f"\n{'='*50}")
    print(f"Evaluating {name}")
    print(f"{'='*50}")
    clf.fit(X_train, y_train)
    train_acc = clf.score(X_train, y_train)
    test_acc = clf.score(X_test, y_test)
    y_pred = clf.predict(X_test)
    print(f"Training Accuracy: {train_acc:.4f}")
    print(f"Test Accuracy: {test_acc:.4f}")
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot()
    plt.title(f"Confusion Matrix: {name}")
    plt.show()
    print('Classification Report:')
    print(classification_report(y_test, y_pred))
    acc = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
    recall = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
    metrics_table.append({
        'Classifier': name,
        'Accuracy': acc,
        'Precision': precision,
        'Recall': recall,
        'F1': f1
    })
    results[name] = {'train_accuracy': train_acc, 'test_accuracy': test_acc}

metrics_df = pd.DataFrame(metrics_table)
print("\nAverage Metrics for Each Classifier:")
print(metrics_df)

# Plot comparison of all classifier results
names = list(results.keys())
train_accuracies = [results[name]['train_accuracy'] for name in names]
test_accuracies = [results[name]['test_accuracy'] for name in names]

x = np.arange(len(names))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
rects1 = ax.bar(x - width/2, train_accuracies, width, label='Training Accuracy')
rects2 = ax.bar(x + width/2, test_accuracies, width, label='Test Accuracy')

ax.set_xlabel('Classifiers')
ax.set_ylabel('Accuracy')
ax.set_title('Classifier Performance Comparison')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=45)
ax.legend()

# Add value labels on bars
def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.3f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom')

autolabel(rects1)
autolabel(rects2)

plt.tight_layout()
plt.show()
